# CLARIN subset review

Browse the **132 selected fragments** from
`~/datasets/clarin_all_2speaker_fragments/manifest.csv`. Per fragment you
get a header (index + frag_id + key stats), a diarization plot showing
the fragment delimited by dashed lines with ±5 s of context, and an audio player.

**Audio is sliced on the fly** from the source recordings under
`~/datasets/clarin_all_2speakers/clarin_download/` — no extracted WAVs needed.

Use the **`START_INDEX` / `LIMIT`** knobs in the page cell to browse in
chunks of 20.

In [ ]:
from IPython.display import HTML, display
display(HTML("<style>audio{width:100% !important;}</style>"))  # szersze odtwarzacze audio

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd() if Path('scripts').is_dir() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display, Audio, Markdown
import soundfile as sf

from scripts.clarin_fragment_finder import load_diarization, overlap_intervals

SUBSET_ROOT  = Path('~/datasets/clarin_all_2speaker_fragments').expanduser()
SOURCE_ROOT  = Path('~/datasets/clarin_all_2speakers').expanduser()
AUDIO_DIR    = SOURCE_ROOT / 'clarin_download'
DIAR_DIR     = SOURCE_ROOT / 'diarization'

MANIFEST = SUBSET_ROOT / 'manifest.csv'
manifest = pd.read_csv(MANIFEST)
print(f'loaded manifest: {len(manifest)} fragments, '
      f'{manifest["rec"].nunique()} source recordings, '
      f'{manifest["duration"].sum()/60:.1f} min total')
print()
print('cell counts:')
print(pd.crosstab(manifest['overlap_bin'], manifest['noise']))

## Helpers

- `plot_diarization` — per-speaker activity bars + red overlap shading; if
  a `fragment` is passed, marks its boundaries with two dashed vertical lines.
- `get_audio(rec)` — lazy-loads the source recording once per session and
  caches it. With ~25 source recordings per page at ~25 MB each, the cache
  grows to ~500 MB after a few pages. Call `audio_cache.clear()` if you
  need to reclaim memory.
- `get_diarization(rec)` — same caching for the diarization JSONs (tiny).

In [ ]:
audio_cache = {}
diar_cache  = {}

def get_audio(rec):
    if rec not in audio_cache:
        wav_path = AUDIO_DIR / f'{rec}.wav'
        if not wav_path.exists():
            audio_cache[rec] = (None, None)
        else:
            arr, sr = sf.read(str(wav_path), dtype='float32', always_2d=False)
            if arr.ndim > 1:
                arr = arr.mean(axis=1)
            audio_cache[rec] = (arr, sr)
    return audio_cache[rec]

def get_diarization(rec):
    if rec not in diar_cache:
        path = DIAR_DIR / f'{rec}.json'
        if not path.exists():
            diar_cache[rec] = ([], 0.0, [])
        else:
            turns, total = load_diarization(path)
            overlaps = overlap_intervals(turns)
            diar_cache[rec] = (turns, total, overlaps)
    return diar_cache[rec]

def plot_diarization(turns, total_dur, overlaps, fragment=None,
                     xlim=None, title=None, figsize=(13, 2.2)):
    """Per-speaker bars + overlap highlights. If `fragment=(start, end)` is
    given, draws two black dashed vertical lines at its boundaries (used
    when the plot is zoomed to fragment \u00b1 context and we just need
    to mark where the fragment proper begins and ends).
    """
    speakers = sorted({t.speaker for t in turns})
    fig, ax = plt.subplots(figsize=figsize)
    y_for = {spk: i for i, spk in enumerate(speakers)}
    for t in turns:
        ax.add_patch(Rectangle(
            (t.start, y_for[t.speaker] - 0.3), t.end - t.start, 0.6,
            facecolor='C0', edgecolor='none', alpha=0.7,
        ))
    for os_, oe in overlaps:
        ax.axvspan(os_, oe, color='red', alpha=0.15)
    if fragment is not None:
        fs, fe = fragment
        ax.axvline(fs, color='black', linestyle='--', linewidth=1.0, alpha=0.7)
        ax.axvline(fe, color='black', linestyle='--', linewidth=1.0, alpha=0.7)
    ax.set_yticks(range(len(speakers)))
    ax.set_yticklabels(speakers)
    ax.set_xlim(xlim or (0, total_dur))
    ax.set_ylim(-0.8, len(speakers) - 1 + 0.8)
    ax.set_xlabel('time (s)')
    if title:
        ax.set_title(title, fontsize=10)
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

## Page

Edit `START_INDEX` to move through the 132 fragments. Default page size is 20.

- `START_INDEX = 0`   → fragments 0–19
- `START_INDEX = 20`  → fragments 20–39
- `START_INDEX = 120` → fragments 120–131 (last page, 12 items)

In [ ]:
# --- Knobs ---
START_INDEX = 0
LIMIT       = 20
PAD_S       = 5.0    # context shown on either side of the fragment in the plot

end_index = min(START_INDEX + LIMIT, len(manifest))
page = manifest.iloc[START_INDEX:end_index]
print(f'showing fragments {START_INDEX}..{end_index - 1}  '
      f'(of 0..{len(manifest) - 1}, {len(manifest)} total)')
print()

for i, row in page.iterrows():
    # Header — index FIRST, then everything else
    display(Markdown(
        f"### **{i}**  ·  `{row['frag_id']}`  ·  "
        f"{row['overlap_bin']}/{row['noise']}  ·  "
        f"score = {row['score']:.2f}"
    ))
    print(
        f"  rec      : {row['rec']}\n"
        f"  time     : [{row['start']:7.2f} → {row['end']:7.2f}]  ({row['duration']:.1f}s)\n"
        f"  overlap  : {row['overlap_s']:.1f}s ({row['overlap_s']/row['duration']*100:.1f}%)  "
        f"events={row['n_overlap_events']}\n"
        f"  balance  : {row['speaker_balance']:.2f}  "
        f"speech_density = {row['speech_density']:.2f}\n"
        f"  noise    : {row['Poziom szumów']}  ({row['Typ szumów']})\n"
        f"  env      : {row['Środowisko']}  ·  {row['Urządzenie nagrywające']}  ·  {row['Typ mikrofonu']}\n"
        f"  topic    : {row['Temat rozmowy']}\n"
        f"  style    : {row['Styl']}"
    )

    # Diarization plot (fragment + PAD_S context, fragment bounds marked with dashed lines)
    turns, total_dur, overlaps = get_diarization(row['rec'])
    if turns:
        plot_diarization(
            turns, total_dur, overlaps,
            fragment=(row['start'], row['end']),
            xlim=(max(0, row['start'] - PAD_S),
                  min(total_dur, row['end'] + PAD_S)),
            title=f"#{i} — fragment between the dashed lines, ±{PAD_S:.0f}s context",
        )
    else:
        print('  (diarization missing)')

    # Audio player — slice from the lazily-loaded source
    audio, sr = get_audio(row['rec'])
    if audio is not None:
        lo = int(row['start'] * sr)
        hi = int(row['end']   * sr)
        display(Audio(audio[lo:hi], rate=sr))
    else:
        print('  (source audio missing)')

    print()

## Notes

- The diarization plot only shows the fragment + ±5 s of context (set
  `PAD_S = 0` for an exact-fragment view, or larger for more context).
  The dashed lines mark the actual fragment boundaries.
- The page cell re-renders only the slice you ask for. After loading
  pages 0 and 1 you have ~40 source recordings cached in RAM (~1 GB);
  call `audio_cache.clear(); diar_cache.clear()` between pages if you
  need to reclaim it.
- If a fragment sounds wrong (no overlap, both speakers very quiet,
  weird cut), note its index here and tell me — I can adjust
  `FragmentParams` and re-run `scripts/build_clarin_fragment_set.py`.